**1. Import Packages**

In [185]:
# Data Manipulation Packages
import pandas as pd
import numpy as np
from pandasql import sqldf
from datetime import datetime, timedelta

# Report Generation
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.pdfgen.canvas import Canvas
from reportlab.platypus.frames import Frame

# Directory Packages
from functools import partial
import os
from pathlib import Path
from dotenv import load_dotenv
import glob

# Package To Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

**2. Create Paths**

In [186]:
ROOT_DIR: Path = Path().resolve().parent
load_dotenv(os.path.join(ROOT_DIR, ".env"))
Root = os.path.normpath(os.getcwd() + os.sep + os.pardir)

In [187]:
footer      = Root+'/Images/Footer.jpg'
header      = Root+'/Images/Header.jpg'
data_path   = Root+"/Data"

**3. Import CSV To Dataframes**

In [188]:
dataframes = {}

for file in glob.glob(os.path.join(data_path, "*.csv")):
    name = os.path.splitext(os.path.basename(file))[0]  # file name without .csv
    dataframes[name] = pd.read_csv(file)

**4. Join All Dataframes**

In [189]:
df = pd.merge(dataframes['sales'], dataframes['products'],on='product_id', how='left')
df = pd.merge(df, dataframes['customers'],on='customer_id', how='left')
df = pd.merge(df, dataframes['stores'],on='store_id', how='left')

In [190]:
df['order_month'] = df['order_date'].str[:7]
df['order_year'] = df['order_date'].str[:4]

**5. Query Data Using SQL**

In [191]:
winners = """with revenue AS 
                (SELECT customer_id
                        ,order_month
                        ,SUM(revenue) AS total_revenue
                        ,SUM(quantity) AS total_units
                        ,SUM(profit) AS total_profit
                FROM df
                GROUP BY 1,2
                )
         SELECT  customer_id
                ,order_month
                ,total_revenue
                ,total_units
                ,total_profit
                ,RANK () OVER(PARTITION BY order_month ORDER BY total_profit DESC) AS Rnk
         FROM revenue
        """

df_win = sqldf(winners)

**6. Clean Data**

In [ ]:
df_win = df_win[(df_win['Rnk'] >= 1) & (df_win['Rnk'] <= 10) & (df_win['order_month'] == df_win['order_month'].max())].sort_values(by=['Rnk'], ascending=True)

In [180]:
df_win = df_win.rename(columns={"customer_id": "Customer ID"
                               ,"order_month": "Order Month"
                               ,"total_revenue": "Total Revenue"
                               ,"total_units": "Total Units"
                               ,"total_profit": "Total Profit"
                               ,"Rnk": "#"}
                        )

In [181]:
df_win['Total Revenue'] = df_win['Total Revenue'].apply(lambda x: "{:,.2f}".format(x))
df_win['Total Units'] = df_win['Total Units'].apply(lambda x: "{:,.2f}".format(x))
df_win['Total Profit'] = df_win['Total Profit'].apply(lambda x: "{:,.2f}".format(x))

In [182]:
df_win = df_win[["Order Month","#","Customer ID","Total Revenue","Total Units","Total Profit"]]
Report = [df_win.columns.to_list()]

**7. Export PDF**

In [183]:
def create_report(x):
    t1 = Table(x)

    t1.setStyle(TableStyle([
        ('INNERGRID', (0, 0), (-1, -1), 0.05, colors.black),
        ('BOX', (0, 0), (-1, -1), 0.05, colors.black),
        ('BACKGROUND', (0, 0), (9, 0), colors.black),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('FONTSIZE', (0, 0), (-1, 0), 10),
        ('FONTSIZE', (0, 1), (-1, -1), 10)
    ]))

    winners = [
        Spacer(30, 30),
        Paragraph("Top 10 Customers By Revenue", getSampleStyleSheet()['Heading1']),
        t1
    ]


    c = Canvas('Top 10 Customers By Revenue.pdf', pagesize=letter)
    width, height = letter

    # Draw header and footer images
    c.drawInlineImage(header, 0, height - 1.4 * inch, width=width, height=1.4 * inch)
    c.drawInlineImage(footer, 0, 0, width=width, height=0.7 * inch)

    # Create frame for first page and add content
    f1 = Frame(inch, inch, 6 * inch, 9 * inch)
    f1.addFromList(winners, c)


    c.save()

In [184]:
create_report(Report)